# Cross-source dashboard (Phase 5b)

This notebook uses **`analytics.load_unified`** (Phase 5a) to pull **national official statistics** (DCCEEW Australia, PPAC India) and **JODI** into one table, then charts comparisons in a common unit (**kbd** by default).

## What you should already have

- `data/processed/australia/australia_petroleum_statistics.parquet` (with `product_canonical` / `category` from Phase 4d)
- `data/processed/india/india_pt_consumption.parquet` (Phase 4b)
- `data/processed/jodi/jodi_secondary.parquet` (Phase 4c)

## Sections

1. **Setup** — project root, imports, `load_unified`
2. **Tangible demo** — one month, diesel, multiple countries & sources side-by-side
3. **Column cheat sheet** — how `product_map.csv` maps to parquet columns
4. **India: PPAC vs JODI** — diesel demand time series (`cross_source_comparison_chart`)
5. **Australia: DCCEEW vs JODI** — diesel demand time series
6. **Seasonality (canonical product)** — `seasonality_by_year_chart` on unified data in **kbd**
7. **TODO (your turn)** — ideas to extend the analysis

## Unit notes (read once)

- JODI **`KL`** is treated as **ML** (megalitres) inside `load_unified` — see `analytics/unified.py` docstring.
- JODI **`CONVBBL`** rows get **NaN** `value_target` until we have a documented conversion; prefer **KBBL** or **KBD** for barrel-based lines.

## 1. Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from IPython.display import display


def resolve_project_root() -> Path:
    """Find `country_oil_scraper/` whether the notebook cwd is repo root or `notebooks/`."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_australia.py").exists():
            return candidate
        nested = candidate / "country_oil_scraper"
        if (nested / "scripts" / "update_australia.py").exists():
            return nested
    raise RuntimeError(f"Could not find country_oil_scraper from cwd={here}")


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics import load_unified, clear_unified_caches
from analytics.charts import cross_source_comparison_chart, seasonality_by_year_chart

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)

PROJECT_ROOT: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper
PROCESSED_ROOT: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed


## 2. Tangible demo — one month, diesel, multiple sources

Same **canonical** product (`product_canonical == "Diesel"`), different **native** labels (`HSD`, `GASDIES`, `Diesel oil: total`).

In [12]:
demo = load_unified(
    metric="TOTDEMO",
    product_canonical="Diesel",
    countries=["Australia", "India"],
    start="2024-03-01",
    end="2024-03-01",
    processed_dir=PROCESSED_ROOT,
)
cols = [
    "country",
    "source",
    "product_native",
    "product_canonical",
    "category",
    "metric_type",
    "unit_native",
    "value_native",
    "value_target",
]
display(demo[cols].sort_values(["source", "country", "unit_native"]))

,country,source,product_native,product_canonical,category,metric_type,unit_native,value_native,value_target
0,Australia,DCCEEW,Diesel oil: total,Diesel,Distillates,TOTDEMO,ML,2764.8,560.969893
1,Australia,JODI,GASDIES,Diesel,Distillates,TOTDEMO,CONVBBL,7460.0,<NA>
2,Australia,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KBBL,17419.1,561.906452
3,Australia,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KBD,561.9065,561.9065
4,Australia,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KL,2769.4203,561.907339
5,Australia,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KTONS,2335.0,561.906452
6,India,JODI,GASDIES,Diesel,Distillates,TOTDEMO,CONVBBL,7458.0,<NA>
7,India,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KBBL,60454.548,1950.14671
8,India,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KBD,1950.1467,1950.1467
9,India,JODI,GASDIES,Diesel,Distillates,TOTDEMO,KL,9611.5215,1950.149808


## 3. Column cheat sheet (CSV ↔ parquet)

| `product_map.csv` | Parquet column | Meaning |
|-------------------|----------------|---------|
| `Product_name` | `product_native` / `product` / `energy_product` | Agency-specific label |
| `Sub-category` | **`product_canonical`** | Shared bucket (e.g. Diesel, Gasoline) |
| `Category` | **`category`** | Broader group (e.g. Distillates) |

The rename is intentional: pandas-friendly names + `product_canonical` reads clearly next to the native `product` column. Full rationale lives in the `#` header at the top of `reference/product_map.csv` and in `reference/loaders.py`.

## 4. India — PPAC vs JODI (diesel, monthly, kbd)

We pick **one JODI unit** (`KBBL`) so the orange line is not duplicated five times. PPAC is already in **kt**; `load_unified` converts both sides to **kbd**.

In [21]:
india = load_unified(
    metric="TOTDEMO",
    product_canonical=["Diesel","Gasoline"],
    countries=["India"],
    start="2015-01-01",
    end=None,
    processed_dir=PROCESSED_ROOT,
)

df_ppac = india[india["source"] == "PPAC"].copy()
df_jodi = india[(india["source"] == "JODI") & (india["unit_native"] == "KBBL")].copy()

# `cross_source_comparison_chart` needs the SAME product string in both frames.
# We use the canonical name on both sides.
df_ppac["plot_product"] = df_ppac["product_canonical"]
df_jodi["plot_product"] = df_jodi["product_canonical"]

fig_in = cross_source_comparison_chart(
    df_ppac,
    df_jodi,
    products=["Diesel", "Gasoline"],
    label_a="PPAC (India)",
    label_b="JODI India (KBBL → kbd)",
    value_col_a="value_target",
    value_col_b="value_target",
    date_col_a="date",
    date_col_b="date",
    product_col_a="plot_product",
    product_col_b="plot_product",
    title="India — diesel apparent demand: national (PPAC) vs JODI",
    units_label="kbd (thousand barrels per day)",
)
fig_in.show()

C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\analytics\unified.py:432: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)


In [20]:
df_ppac['product_canonical'].unique()

array(['Diesel'], dtype=object)

## 5. Australia — DCCEEW vs JODI (diesel, monthly, kbd)

Same pattern: DCCEEW official sales vs JODI secondary, **JODI unit = KBBL** for a single comparable line.

In [ ]:
au = load_unified(
    metric="TOTDEMO",
    product_canonical=["Diesel","Gasoline"],
    countries=["Australia"],
    start="2015-01-01",
    end=None,
    processed_dir=PROCESSED_ROOT,
)

df_dceew = au[au["source"] == "DCCEEW"].copy()
df_jodi_au = au[(au["source"] == "JODI") & (au["unit_native"] == "KBBL")].copy()

df_dceew["plot_product"] = df_dceew["product_canonical"]
df_jodi_au["plot_product"] = df_jodi_au["product_canonical"]

fig_au = cross_source_comparison_chart(
    df_dceew,
    df_jodi_au,
    products=["Diesel", "Gasoline"],
    label_a="DCCEEW (Australia)",
    label_b="JODI Australia (KBBL → kbd)",
    value_col_a="value_target",
    value_col_b="value_target",
    date_col_a="date",
    date_col_b="date",
    product_col_a="plot_product",
    product_col_b="plot_product",
    title="Australia — diesel demand: DCCEEW vs JODI",
    units_label="kbd (thousand barrels per day)",
)
fig_au.show()

## 6. Seasonality by calendar year (canonical **Gasoline**, PPAC India, kbd)

Uses `seasonality_by_year_chart` with **`value_target`** and **`product_canonical`** as the product column so the API sees stable labels.

In [ ]:
in_gas = load_unified(
    metric="TOTDEMO",
    product_canonical=["Gasoline"],
    countries=["India"],
    sources=["PPAC"],
    start="2018-01-01",
    end=None,
    processed_dir=PROCESSED_ROOT,
)

fig_seas = seasonality_by_year_chart(
    in_gas,
    products=["Gasoline"],
    value_col="value_target",
    date_col="date",
    product_col="product_canonical",
    title="India PPAC — gasoline demand seasonality by calendar year (kbd)",
    units_label="kbd",
)
fig_seas.show()

## 7. TODO — your turn

- Add a **second product** to the India PPAC vs JODI chart (e.g. `Gasoline`) — `products=["Diesel", "Gasoline"]` once both sides have rows for each.
- Compare **JODI `KBD`** (native rate) vs **kbd** from monthly stocks — discuss which is closer to PPAC's monthly deliveries.
- After refreshing parquets, run **`clear_unified_caches()`** once so `load_unified` re-reads disk.
- Extend `load_unified` filters (`category=`, multiple `metric=` codes) for a **supply-side** view (`TOTIMPSB`, `REFGROUT`).
- Map more Australia `product_native` labels in `product_map.csv` if you want **non-Sales** sheets in cross-source charts.

In [44]:
au_gas = load_unified(
    metric="TOTDEMO",
    product_canonical=["Gasoline", 'Diesel', 'Fuel Oil', 'Jet Fuel'],
    countries=["India"],
    sources=["PPAC"],
    start="2018-01-01",
    end=None,
    processed_dir=PROCESSED_ROOT,
)

fig_seas = seasonality_by_year_chart(
    au_gas,
    products=["Gasoline", 'Diesel', 'Fuel Oil', 'Jet Fuel'],
    value_col="value_target",
    date_col="date",
    product_col="product_canonical",
    title="AU DCCEEW — gasoline demand seasonality by calendar year (kbd)",
    units_label="kbd",
)
fig_seas.show()